# Assignment 5: k-Means and expectation maximization

k-Means comes from scikit-learn's `KMeans` class, and expectation maximization (EM) for a Gaussian
mixture model comes from its `GaussianMixture` class. You do not implement either algorithm. Your
job is to configure both models, read the learned parameters out of each fitted model, choose the
number of customer segments, use k-Means to compress images, and write the report.

The notebook has three parts: synthetic Gaussian mixtures, where the true clusters are known;
customer segmentation on real sales data, where they are not; and image compression. The datasets,
the repeated runs, the scoring, and the plots are provided. Every synthetic experiment is repeated
50 times on freshly sampled data, and the results are averages over those runs.

Six code cells are yours: `build_kmeans`, `build_gmm`, `kmeans_means`, `gmm_parameters`, the number
of customer segments, and `compress_image`. Everything else is provided and already runs.

Jake VanderPlas's Python Data Science Handbook has a notebook on each algorithm:
[k-Means](https://github.com/jakevdp/PythonDataScienceHandbook/blob/master/notebooks/05.11-K-Means.ipynb)
and [Gaussian Mixtures](https://github.com/jakevdp/PythonDataScienceHandbook/blob/master/notebooks/05.12-Gaussian-Mixtures.ipynb).

# Running in Colab

Open this notebook from its Colab link in the spec, then save your own copy right away with
**File > Save a copy in Drive**. Work in that copy, or your changes are lost when Colab closes.
This notebook does not need a GPU.

The next cell copies the course repository onto the Colab machine so the notebook finds its data in
`../data`. Run it first every time you open the notebook. On your own computer it does nothing.

In [ ]:
import os, sys, subprocess

# in colab, fetch the course repository so ../data exists
if "google.colab" in sys.modules:
    if not os.path.exists("/content/fund-ml-project-5"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/zacharytgray/fund-ml-project-5",
                        "/content/fund-ml-project-5"], check=True)
    os.chdir("/content/fund-ml-project-5/assignment")

# Import libraries
Do not use any other Python library.

numpy - Linear algebra library for handling vectors and matrices, collectively processed as numpy arrays.

matplotlib - Graphing library for visualizing results.

glob - Lists the image files in a folder.

sklearn - Machine learning library from which we source the k-Means model, the Gaussian mixture model, the Rand index, and the silhouette score.

skimage - Image library used to read the image files and convert their pixels to numbers from 0 to 1.

os, sys, subprocess - Used by the Colab setup cell above, and `os` to print file names.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from glob import glob
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import rand_score, silhouette_score
from skimage.io import imread
from skimage.util import img_as_float

# Synthetic Gaussian mixtures

# Function: generate_mixture

This function is provided. It samples a 1-dimensional dataset of $S \cdot N$ points: $N$ points from
each of $S$ Gaussian sources. The source means are $0, \Delta\mu, 2\Delta\mu, \ldots,
(S-1)\Delta\mu$. Each source has its own standard deviation $\sigma_i$: pass one number to give
every source the same $\sigma$, or a list of $S$ numbers.

Inputs:
*   **S**: number of sources (integer)
*   **N**: number of points from each source (integer)
*   **delta_mu**: spacing $\Delta\mu$ between neighboring source means (float)
*   **sigma**: standard deviation of every source (float), or a list of $S$ standard deviations
*   **rng**: a numpy random generator, so each run draws its own data

Outputs:
*   **X**: numpy array of shape ($S \cdot N$, 1), the data points
*   **y**: numpy array of $S \cdot N$ integer source labels. The models never see these; they are used only to score the clusters
*   **means**, **sigmas**: numpy arrays of the $S$ true means and standard deviations

# Functions: plot_kmeans_fit and plot_gmm_fit

These are provided. Each draws a histogram of the data with one color per true source, scaled so
the bars of all sources together have area 1. `plot_kmeans_fit` marks each k-Means cluster center
with a dashed line. `plot_gmm_fit` draws each fitted Gaussian component, multiplied by its mixture
weight, and the mixture density, which is their sum, on the same scale as the histogram.

In [ ]:
def generate_mixture(S, N, delta_mu, sigma, rng):
    means = delta_mu * np.arange(S, dtype=float)
    # one sigma for every source, or one per source
    sigmas = np.broadcast_to(np.asarray(sigma, dtype=float), (S,)).copy()
    X = np.concatenate([rng.normal(means[s], sigmas[s], N) for s in range(S)]).reshape(-1, 1)
    y = np.repeat(np.arange(S), N)
    return X, y, means, sigmas


def plot_sources(ax, X, y):
    # density histogram per source; together the sources integrate to 1
    bins = np.linspace(X.min(), X.max(), 60)
    width = bins[1] - bins[0]
    for s in np.unique(y):
        points = X[y == s, 0]
        ax.hist(points, bins=bins, weights=np.full(len(points), 1 / (len(X) * width)), alpha=0.4)
    ax.set_xlabel("x")
    ax.set_ylabel("density")


def plot_kmeans_fit(ax, X, y, centers, title):
    plot_sources(ax, X, y)
    for i, center in enumerate(centers):
        ax.axvline(center, color="black", linestyle="--", linewidth=1,
                   label="cluster centers" if i == 0 else None)
    ax.set_title(title)
    ax.legend()


def plot_gmm_fit(ax, X, y, means, stds, weights, title):
    plot_sources(ax, X, y)
    x = np.linspace(X.min(), X.max(), 1000)
    total = np.zeros_like(x)
    for mean, std, weight in zip(means, stds, weights):
        pdf = weight * np.exp(-0.5 * ((x - mean) / std) ** 2) / np.sqrt(2 * np.pi * std ** 2)
        total += pdf
        ax.plot(x, pdf, linewidth=1)
    ax.plot(x, total, color="black", linestyle="--", label="mixture density")
    ax.set_title(title)
    ax.legend()

# Function: build_kmeans

Builds one `KMeans` model with C clusters, not yet fitted. Where k-Means ends up depends on its
starting centers, so run several restarts from different starting centers; the model keeps the
best one. Pass the seed as the model's random state, so every run can be repeated exactly. Look up
the [`KMeans`](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html)
constructor to find which arguments set the number of clusters, the number of restarts, and the
random state.

Inputs:
*   **C**: number of clusters (integer)
*   **seed**: random seed (integer)

Output:
*   **model**: an unfitted `KMeans`

In [ ]:
def build_kmeans(C, seed):
    # 1. build k-means with C clusters, several restarts, and the seed as its random state
    # TODO
    pass

# Function: build_gmm

Builds one `GaussianMixture` model with C components, not yet fitted. It is trained with EM. Each
component must be allowed its own variance, so choose a covariance type that does not share one
covariance across all components. EM stops once an iteration improves the log-likelihood by less
than a tolerance. Set a tolerance of about $10^{-5}$ and raise the iteration limit, so EM runs until
it converges. Pass the seed as the random state. Look up the
[`GaussianMixture`](https://scikit-learn.org/stable/modules/generated/sklearn.mixture.GaussianMixture.html)
constructor to find which arguments set the number of components, the covariance type, the
tolerance, the iteration limit, and the random state.

Inputs:
*   **C**: number of components (integer)
*   **seed**: random seed (integer)

Output:
*   **model**: an unfitted `GaussianMixture`

In [ ]:
def build_gmm(C, seed):
    # 1. build a gaussian mixture with C components, per-component variances, a tighter tolerance,
    #    a higher iteration limit, and the seed as its random state
    # TODO
    pass

# Function: kmeans_means

Returns the cluster centers of a fitted `KMeans` model as a 1-dimensional numpy array of length C.
The data has one feature, so each center is a single number. The order does not matter; the
experiment code sorts them.

Input:
*   **model**: a fitted `KMeans`

Output:
*   **centers**: numpy array of C numbers

In [ ]:
def kmeans_means(model):
    # 1. read the fitted cluster centers and return them as one number per cluster
    # TODO
    pass

# Function: gmm_parameters

Returns the mean and the standard deviation of every component of a fitted `GaussianMixture`, as
two 1-dimensional numpy arrays of length C, in the same component order. The model stores
variances, not standard deviations, and the shape of its covariance array depends on the
covariance type you chose in `build_gmm`. The experiment code sorts the components by their means.

Input:
*   **model**: a fitted `GaussianMixture`

Outputs:
*   **means**: numpy array of C numbers
*   **stds**: numpy array of C numbers

In [ ]:
def gmm_parameters(model):
    # 1. read the component means as one number per component
    # 2. read each component's variance from the covariance array and turn it into a standard deviation
    # TODO
    pass

# Function: run_experiment

This function is provided. It runs one row of an experiment for one method, `"kmeans"` or `"em"`.
For each of the `runs` repetitions it samples a new dataset with `generate_mixture`, builds a model
with your `build_kmeans` or `build_gmm` from the same seed, fits it, and assigns every point to a
cluster. Run $r$ uses seed $r$ for both the data and the model, so k-Means and EM see the same 50
datasets.

The cluster numbers a model picks are arbitrary: its cluster 0 need not be source 0. So the
clusters are scored with the
[Rand index](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.rand_score.html),
which looks at pairs of points. A pair is counted as agreeing when the two points share a source
and also share a cluster, or come from different sources and also land in different clusters. The
Rand index is the fraction of all pairs that agree, from 0 to 1, and it does not depend on how the
clusters are numbered.

When C = S, it also keeps each run's learned means, sorted, and for EM the standard deviations in
the same order as the means, so they can be set against the true source parameters. It keeps the
first run's data and fit for the plots.

Inputs:
*   **method**: `"kmeans"` or `"em"`
*   **S**, **N**, **delta_mu**, **sigma**: the dataset settings, as in `generate_mixture`
*   **C**: number of clusters or components (integer)
*   **runs**: number of repetitions (integer)

Output:
*   **result**: dictionary with the Rand index of every run, the first run's data and fit, and, when C = S, the sorted learned parameters of every run

# Function: print_result

This function is provided. It prints the mean and standard deviation of the Rand index over the
runs and, when C = S, a table of the true parameters against the learned ones: the mean over the
runs, plus or minus the standard deviation over the runs.

In [ ]:
def run_experiment(method, S, N, delta_mu, sigma, C, runs):
    result = {"rand": [], "means": [], "stds": []}
    for run in range(runs):
        rng = np.random.default_rng(run)
        X, y, true_means, true_sigmas = generate_mixture(S, N, delta_mu, sigma, rng)
        if method == "kmeans":
            model = build_kmeans(C, run)
            model.fit(X)
            means, stds, weights = kmeans_means(model), None, None
        else:
            model = build_gmm(C, run)
            model.fit(X)
            means, stds = gmm_parameters(model)
            weights = model.weights_
        result["rand"].append(rand_score(y, model.predict(X)))

        # sort the learned clusters by mean so they line up with the sources
        order = np.argsort(means)
        means = means[order]
        if stds is not None:
            stds, weights = stds[order], weights[order]
        if C == S:
            result["means"].append(means)
            if stds is not None:
                result["stds"].append(stds)
        if run == 0:
            result["first"] = {"X": X, "y": y, "means": means, "stds": stds, "weights": weights}

    result["true_means"], result["true_sigmas"] = true_means, true_sigmas
    return result


def print_result(method, S, sigma, C, result):
    rand = np.array(result["rand"])
    print(f"{method:6} S={S:<3} sigma={sigma:<4} C={C:<3} Rand index {rand.mean():.4f} ± {rand.std():.4f}")
    if C != S:
        return
    means = np.array(result["means"])
    header = f"    {'source':>6} {'true mean':>10} {'learned mean':>18}"
    if result["stds"]:
        stds = np.array(result["stds"])
        header += f" {'true sigma':>11} {'learned sigma':>18}"
    print(header)
    for s in range(S):
        line = (f"    {s:6d} {result['true_means'][s]:10.3f} "
                f"{means[:, s].mean():9.3f} ± {means[:, s].std():6.3f}")
        if result["stds"]:
            line += (f" {result['true_sigmas'][s]:11.3f} "
                     f"{stds[:, s].mean():9.3f} ± {stds[:, s].std():6.3f}")
        print(line)

# Experiment settings

The runs the spec asks for, each with $N = 1000$ points per source and mean spacing
$\Delta\mu = 2$. Every row is run 50 times with k-Means and 50 times with EM.

1.  Number of sources: $S \in \{3, 5, 10\}$, $\sigma = 1$, $C = S$
2.  Size of the sources: $S = 5$, $\sigma \in \{1, 2, 3\}$, $C = S$
3.  Number of clusters: $S = 3$, $\sigma = 1$, $C \in \{2, 3, 6, 8\}$

# Function: run_set

This function is provided. It runs every row of one experiment with both methods, prints the
results, and plots the first run's k-Means fit beside the first run's EM fit. A row that appears in
two experiments is run once and its results are reused.

In [ ]:
N_RUNS = 50
EXPERIMENTS = {
    1: [{"S": S, "N": 1000, "delta_mu": 2.0, "sigma": 1.0, "C": S} for S in [3, 5, 10]],
    2: [{"S": 5, "N": 1000, "delta_mu": 2.0, "sigma": sigma, "C": 5} for sigma in [1.0, 2.0, 3.0]],
    3: [{"S": 3, "N": 1000, "delta_mu": 2.0, "sigma": 1.0, "C": C} for C in [2, 3, 6, 8]],
}

finished = {}
results = []


def run_set(number):
    for row in EXPERIMENTS[number]:
        fig, axes = plt.subplots(1, 2, figsize=(14, 4))
        for ax, method in zip(axes, ["kmeans", "em"]):
            key = (method, row["S"], row["N"], row["delta_mu"], row["sigma"], row["C"])
            if key not in finished:
                finished[key] = run_experiment(method, row["S"], row["N"], row["delta_mu"], row["sigma"],
                                               row["C"], N_RUNS)
            result = finished[key]
            print_result(method, row["S"], row["sigma"], row["C"], result)
            results.append({"experiment": number, "method": method, **row, "rand": result["rand"]})

            first = result["first"]
            title = f"{method}: S={row['S']}, sigma={row['sigma']}, C={row['C']} (run 0)"
            if method == "kmeans":
                plot_kmeans_fit(ax, first["X"], first["y"], first["means"], title)
            else:
                plot_gmm_fit(ax, first["X"], first["y"], first["means"], first["stds"], first["weights"], title)
        print()
        plt.tight_layout()
        plt.show()

# Experiment 1: number of sources

In [ ]:
run_set(1)

# Experiment 2: size of the sources

In [ ]:
run_set(2)

# Experiment 3: number of clusters

In [ ]:
run_set(3)

# Results table

One row per experiment row and method: the mean and standard deviation of the Rand index over the
50 runs.

In [ ]:
print(f"{'exp':>3} {'method':7} {'S':>3} {'sigma':>6} {'C':>3} {'Rand mean':>10} {'Rand std':>9}")
for r in results:
    print(f"{r['experiment']:3d} {r['method']:7} {r['S']:3d} {r['sigma']:6.1f} {r['C']:3d} "
          f"{np.mean(r['rand']):10.4f} {np.std(r['rand']):9.4f}")

# Customer segmentation

`../data/online_retail_customers.csv` has one row per customer of a UK online gift shop, built from
every sale it made from December 2010 to December 2011 (the UCI
[Online Retail](https://archive.ics.uci.edu/dataset/352/online+retail) dataset). There are 4,338
customers and three columns:

*   **recency_days**: days since the customer's last order, counted from the day after the last order in the data
*   **orders**: number of orders
*   **spend**: total spent, in pounds sterling

There are no labels: nobody knows the true groups of customers, or how many there are. The models
cluster the customers on two features, recency and spend. Both are skewed. The median customer
spent about £670 and a few spent over £100,000. So the code takes the logarithm of each feature and
scales it to mean 0 and standard deviation 1, and the plots use log axes.

In [ ]:
data = np.loadtxt("../data/online_retail_customers.csv", delimiter=",", skiprows=1)
recency, orders, spend = data[:, 1], data[:, 2], data[:, 3]
print(f"{len(data)} customers")

# log of each feature, scaled to mean 0 and standard deviation 1
features = np.column_stack([np.log(recency), np.log(spend)])
feature_mean, feature_std = features.mean(axis=0), features.std(axis=0)
customers = (features - feature_mean) / feature_std

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(recency, spend, s=4, alpha=0.3)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("days since last order")
ax.set_ylabel("total spend (£)")
ax.set_title("customers")
plt.show()

# Choosing the number of clusters

Without labels there is no Rand index. Two other measures are provided, each computed for C = 2 to 8
with your `build_kmeans` and `build_gmm` and seed 0:

*   **Silhouette score**, for k-Means: for each point, how much closer it is to its own cluster than to the nearest other cluster, averaged over all points. It runs from −1 to 1, and higher means tighter, better separated clusters.
*   **BIC** (Bayesian information criterion), for EM: how poorly the mixture fits the data, plus a penalty that grows with the number of parameters. Lower is better. More components always fit the data more closely, so the penalty is what keeps BIC from always favoring more.

In [ ]:
C_VALUES = list(range(2, 9))
silhouettes, bics = [], []
for C in C_VALUES:
    kmeans = build_kmeans(C, 0).fit(customers)
    silhouettes.append(silhouette_score(customers, kmeans.labels_))
    gmm = build_gmm(C, 0).fit(customers)
    bics.append(gmm.bic(customers))
    print(f"C={C}  silhouette {silhouettes[-1]:.3f}  BIC {bics[-1]:.0f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(C_VALUES, silhouettes, marker="o")
axes[0].set_title("k-Means: silhouette score (higher is better)")
axes[1].plot(C_VALUES, bics, marker="o")
axes[1].set_title("EM: BIC (lower is better)")
for ax in axes:
    ax.set_xlabel("number of clusters C")
plt.tight_layout()
plt.show()

# Number of segments

Choose the number of clusters for k-Means and the number of components for EM, using the plots
above. The two need not be the same.

In [ ]:
# 1. choose the number of clusters for k-means and the number of components for em
# TODO
K_KMEANS, K_EM = None, None

# Segments

This cell is provided. It fits k-Means with `K_KMEANS` clusters and EM with `K_EM` components, both
with seed 0, and assigns every customer to a segment. The segments are numbered by their median
spend, lowest first. For each method it prints one row per segment, with the number of customers
and their median recency, orders, and spend, and plots the customers colored by segment, with each
segment's center marked by an ×. Last, it prints the Rand index between the two segmentations: the
fraction of customer pairs that both methods put together, or both put apart.

In [ ]:
kmeans = build_kmeans(K_KMEANS, 0).fit(customers)
gmm = build_gmm(K_EM, 0).fit(customers)
fits = {"k-Means": (kmeans.predict(customers), kmeans.cluster_centers_),
        "EM": (gmm.predict(customers), gmm.means_)}

segments = {}
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (name, (labels, centers)) in zip(axes, fits.items()):
    # number the segments by median spend, lowest first
    order = np.argsort([np.median(spend[labels == c]) for c in range(len(centers))])
    rank = np.empty(len(order), dtype=int)
    rank[order] = np.arange(len(order))
    labels, centers = rank[labels], centers[order]
    segments[name] = labels

    print(f"{name}, {len(centers)} segments")
    print(f"    {'segment':>7} {'customers':>9} {'recency (days)':>14} {'orders':>6} {'spend (£)':>9}")
    for s in range(len(centers)):
        m = labels == s
        print(f"    {s:7d} {m.sum():9d} {np.median(recency[m]):14.0f} {np.median(orders[m]):6.0f} "
              f"{np.median(spend[m]):9.0f}")
    print()

    # centers back in days and pounds
    center_points = np.exp(centers * feature_std + feature_mean)
    for s in range(len(centers)):
        m = labels == s
        ax.scatter(recency[m], spend[m], s=4, alpha=0.4, label=f"segment {s}")
    ax.scatter(center_points[:, 0], center_points[:, 1], marker="x", s=120, color="black", linewidths=3)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("days since last order")
    ax.set_ylabel("total spend (£)")
    ax.set_title(f"{name}: {len(centers)} segments")
    ax.legend(markerscale=4)
plt.tight_layout()
plt.show()

print(f"Rand index between the k-Means and EM segments: {rand_score(segments['k-Means'], segments['EM']):.3f}")

# Image compression

Every pixel of a color image is a point in 3 dimensions: its red, green, and blue values. Clustering
the pixels with k-Means and replacing each pixel by its cluster center redraws the image with only
k colors.

Put 5 images of your choice in the `../data/imgs` folder, as PNG or JPEG files: portraits, famous
paintings, landmarks, or iconic objects. Avoid images made only of a few solid colors, such as
frames from animated shows. One sample image is already there. In Colab, open the **Files** panel
on the left and upload your images into `fund-ml-project-5/data/imgs`; uploads are lost when the
Colab machine shuts down, so keep your own copies.

# Function: load_image

This function is provided. It reads one image file and returns its pixels as numbers from 0 to 1,
in an array of shape (height, width, 3). A transparency (alpha) channel is dropped, and a grayscale
image is copied into all three channels.

# Function: compress_image

Fits a k-Means model with k clusters on a random sample of the image's pixels, then assigns every
pixel of the image to its nearest cluster center and replaces the pixel by that center's color.
The model is fit on the sample and predicts every pixel.

Inputs:
*   **fit_pixels**: a random sample of the image's pixels, shape (M, 3), to fit the model on
*   **pixels**: every pixel of the image, shape (height $\cdot$ width, 3)
*   **k**: number of colors (integer)
*   **seed**: random seed (integer)

Output:
*   **compressed**: numpy array with the shape of `pixels`, each pixel replaced by its cluster center

In [ ]:
def load_image(path):
    image = img_as_float(imread(path))
    # grayscale, with or without alpha, becomes three equal channels
    if image.ndim == 2:
        image = image[:, :, None]
    if image.shape[2] < 3:
        image = np.repeat(image[:, :, :1], 3, axis=2)
    # drop the alpha channel
    return image[:, :, :3]

In [ ]:
def compress_image(fit_pixels, pixels, k, seed):
    # 1. build k-means with k clusters and the seed as its random state, and fit it on the sampled pixels
    # 2. assign every pixel of the image to its nearest cluster center
    # 3. replace each pixel by the color of its center
    # TODO
    pass

# Compress the images

Compresses every image in `../data/imgs` to each number of colors in `K_VALUES`, fitting on a random
sample of `SAMPLE_PIXELS` pixels, and shows the original beside the compressed versions.

In [ ]:
K_VALUES = [3, 5, 10]
SAMPLE_PIXELS = 10000

paths = sorted(p for p in glob("../data/imgs/*") if p.lower().endswith((".png", ".jpg", ".jpeg")))
print(f"{len(paths)} images in ../data/imgs")

for path in paths:
    image = load_image(path)
    pixels = image.reshape(-1, 3)
    rng = np.random.default_rng(0)
    sample = pixels[rng.choice(len(pixels), size=min(SAMPLE_PIXELS, len(pixels)), replace=False)]

    fig, axes = plt.subplots(1, len(K_VALUES) + 1, figsize=(4 * (len(K_VALUES) + 1), 4))
    axes[0].imshow(image)
    axes[0].set_title("original")
    for ax, k in zip(axes[1:], K_VALUES):
        compressed = compress_image(sample, pixels, k, seed=0)
        ax.imshow(np.clip(compressed.reshape(image.shape), 0, 1))
        ax.set_title(f"k = {k} colors")
    for ax in axes:
        ax.axis("off")
    fig.suptitle(os.path.basename(path))
    plt.tight_layout()
    plt.show()

# Report

Cover the following in your report.

*   For each of the three experiments, the Rand index of k-Means and of EM (mean and standard deviation over the 50 runs) for every row, and the fit plots.
*   For every row with C = S, the learned k-Means means, and the learned EM means and standard deviations, against the true source parameters.
*   How the Rand index of each method changes with the number of sources, the size of the sources, and the number of clusters, and where k-Means and EM differ.
*   The silhouette and BIC plots, the number of clusters each plot favors, and the numbers you chose for k-Means and EM, with your reason.
*   The segment tables and plots of both methods, one sentence per segment describing its customers, and where k-Means and EM split the customers differently.
*   Your 5 images compressed to 3, 5, and 10 colors, with a comment on the quality of each compression.